# Lesson 08: The Planning Design Pattern

In this notebook, you will:
1. Define `AgentEnum`, `TravelSubTask`, and `TravelPlan` using `pydantic`.
2. Generate structured plans with `google-genai` (`response_schema=TravelPlan`) and run an iterative re-planning pass.

In [ ]:
!pip install -q -U google-genai pydantic python-dotenv

In [ ]:
import os
from enum import Enum
from typing import List
from pydantic import BaseModel
from google import genai
from google.genai import types

class AgentEnum(str, Enum):
    FlightBooking = "flight_booking"
    HotelBooking = "hotel_booking"
    CarRental = "car_rental"
    ActivitiesBooking = "activities_booking"
    DestinationInfo = "destination_info"

class TravelSubTask(BaseModel):
    task_details: str
    assigned_agent: AgentEnum

class TravelPlan(BaseModel):
    main_task: str
    subtasks: List[TravelSubTask]
    is_greeting: bool

api_key = os.environ.get("GEMINI_API_KEY")
if api_key:
    client = genai.Client(api_key=api_key)
    res = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="Create a travel plan for a family of 2 kids from Singapore to Melbourne.",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=TravelPlan
        )
    )
    print(res.text)
else:
    demo = TravelPlan(
        main_task="Plan a family trip from Singapore to Melbourne.",
        is_greeting=False,
        subtasks=[TravelSubTask(task_details="Book flights SIN->MEL", assigned_agent=AgentEnum.FlightBooking)]
    )
    print(demo.model_dump_json(indent=2))